# מפגש 8 — איך בוחרים K מבלי להטעות את עצמנו?
**יחידה 3 | מחברת מודרכת | Penguins**

הנושאים במצגת ההשתלמות: Underfitting / Overfitting, מורכבות המודל, בחירת K, Train/Validation/Test, Cross Validation, Pipeline ו־GridSearchCV. נתרגל תחילה לולאת K; GridSearchCV היא הרחבת המשך.

## 1. תחזית לפני הרצה
ב־KNN, מספר שכנים קטן יוצר החלטה רגישה לדוגמאות בודדות; מספר שכנים גדול עשוי להחליק יתר על המידה ולהחמיץ דפוסים. אין כלל ש־K הגדול ביותר יצליח הכי טוב.

**שאלה:** האם מודל שמדייק ב־100% על Train בהכרח טוב יותר על Test?

## 2. נתוני Penguins: הכנה וחלוקה
בדוגמה זו מסירים שורות עם מדידות חסרות לצורך פשטות, לפני החלוקה. אין להתאים Scaler על כל הנתונים. נשתמש רק בארבע מדידות מספריות; ננסה לנבא `species`.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

df = pd.read_csv("https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv")
features = ["bill_length_mm","bill_depth_mm","flipper_length_mm","body_mass_g"]
clean = df.dropna(subset=features + ["species"]).copy()
X = clean[features]
y = clean["species"]
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
print("Train:",X_train.shape,"Test:",X_test.shape)

## 3. Accuracy באימון מול בדיקה
לצורך הדגמה בלבד נציג מדדי Train ו־Test עבור מספר ערכי K קבועים. **אין לבחור את K הסופי באמצעות Test**: הוא נשמר להערכה בסוף.

In [ ]:
for k in [1,3,5,15,35]:
    pipe=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=k))
    pipe.fit(X_train,y_train)
    train_score=accuracy_score(y_train,pipe.predict(X_train))
    test_score=accuracy_score(y_test,pipe.predict(X_test))
    print(f'K={k:2d}: Train={train_score:.3f}, Test={test_score:.3f}')

## 4. Validation / Cross Validation
כדי לבחור K מבלי להשתמש ב־Test, נבצע Cross Validation **רק על Train**. בכל קיפול ה־Pipeline מתקן את הסולם לפי חלק האימון של אותו קיפול.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
results=[]
for k in [1,3,5,7,9,15,25]:
    pipe=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=k))
    scores=cross_val_score(pipe,X_train,y_train,cv=cv,scoring='accuracy')
    results.append((k,scores.mean()))
    print(f'K={k:2d}, CV mean={scores.mean():.3f}')
best_k=max(results,key=lambda item:item[1])[0]
print('Best K according to CV:',best_k)

## 5. מבחן סופי אחד
עתה מאמנים מחדש על כל Train בעזרת K שנבחר ב־CV, ומודדים **פעם אחת** על Test.

In [ ]:
final_model=make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=best_k))
final_model.fit(X_train,y_train)
pred=final_model.predict(X_test)
print('Final Test Accuracy:',round(accuracy_score(y_test,pred),3))
print(classification_report(y_test,pred,zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test,pred,cmap='Blues')
plt.show()

## 6. מה למדנו?
- פרמטר שנקבע בתכנות: **K — היפר־פרמטר**.
- הערכת Train עוזרת לזהות התאמה לנתוני האימון.
- Validation/CV משמשים לבחירת K.
- Test משמש להערכה סופית בלבד.

**כרטיס יציאה:** הסבירו מהו Overfitting ומהו Underfitting; למה Pipeline חשוב; ומה יקרה אם נשנה K שוב לאחר שראינו את תוצאות Test.